# Text Representation: One-Hot, BoW, DTM and TF-IDF

Day 1, Session 5. Turning cleaned text into numbers that a model can use, and analysing those numbers

Run this in Google Colab (or any Jupyter setup). Run the cells top to bottom.

## Setup

In [ ]:
# one time installs, colab already has most of these
# nltk         -> movie review corpus for real data
# scikit-learn -> CountVectorizer, TfidfVectorizer, cosine similarity, chi-square
# pandas       -> show matrices as tables
# numpy        -> vector math
%pip install -q nltk scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import nltk

nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Vocabulary

In [ ]:
docs = ["the cat sat", "the dog sat", "the cat chased the dog"]
vocab = sorted({w for d in docs for w in d.split()})
print(vocab, len(vocab))

## 2. One-hot encoding

In [ ]:
idx = {w: i for i, w in enumerate(vocab)}

def one_hot(w):
    v = np.zeros(len(vocab), dtype=int)
    v[idx[w]] = 1
    return v

print(one_hot("cat"), one_hot("dog"))
print("dot product cat . dog =", one_hot("cat") @ one_hot("dog"))
print("dot product cat . cat =", one_hot("cat") @ one_hot("cat"))

## 3. Bag of Words and the document-term matrix

In [ ]:
cv = CountVectorizer()
X = cv.fit_transform(docs)
print(cv.get_feature_names_out())
print(X.toarray())

# same thing as a labelled table, easier to read
print(pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=docs))

In [ ]:
# order is lost: these two sentences get identical rows
pair = ["dog bites man", "man bites dog"]
print(CountVectorizer().fit_transform(pair).toarray())

# bigrams bring back a little order
cv2 = CountVectorizer(ngram_range=(1, 2))
print(cv2.fit_transform(pair).toarray())
print(cv2.get_feature_names_out())

## 4. TF-IDF

A six document collection with two topics, pets and finance.

In [ ]:
docs6 = ["the cat sat on the mat",
         "the dog chased the cat",
         "my dog loves the park",
         "stock prices fell on monday",
         "the market prices rose sharply",
         "investors sold stock as prices fell"]

tv = TfidfVectorizer()
W = tv.fit_transform(docs6)

# weights of document 5 (index 4), highest first
row = zip(tv.get_feature_names_out(), W[4].toarray()[0])
for t, s in sorted(row, key=lambda x: -x[1]):
    if s > 0:
        print(t, round(s, 3))

In [ ]:
# check the formula by hand: idf = ln((1 + n) / (1 + df)) + 1
n = len(docs6)
terms = list(tv.get_feature_names_out())

for word in ["the", "prices", "market"]:
    df = sum(word in d.split() for d in docs6)
    by_hand = np.log((1 + n) / (1 + df)) + 1
    print(word, "df =", df, "idf by hand =", round(by_hand, 4), "sklearn =", round(tv.idf_[terms.index(word)], 4))

## 5. Feature analysis

In [ ]:
sim = cosine_similarity(W)
print(np.round(sim[0], 2))     # document 0 against every document

zeros = 1 - W.nnz / (W.shape[0] * W.shape[1])
print("shape", W.shape, "sparsity", round(100 * zeros, 1), "%")

# similarity heatmap as a table
print(pd.DataFrame(np.round(sim, 2)))

### On real data: movie reviews

Take 100 negative and 100 positive reviews.

In [ ]:
ids = movie_reviews.fileids("neg")[:100] + movie_reviews.fileids("pos")[:100]
texts = [movie_reviews.raw(f) for f in ids]
labels = [0] * 100 + [1] * 100     # 0 = negative, 1 = positive

tv = TfidfVectorizer(stop_words="english", min_df=2)
X = tv.fit_transform(texts)
zeros = 1 - X.nnz / (X.shape[0] * X.shape[1])
print(X.shape, round(100 * zeros, 1))

mean = np.asarray(X.mean(axis=0)).ravel()
top = mean.argsort()[::-1][:8]
print([tv.get_feature_names_out()[i] for i in top])

In [ ]:
# sparse storage vs dense storage
print("sparse bytes:", X.data.nbytes + X.indices.nbytes + X.indptr.nbytes)
print("dense bytes :", X.toarray().nbytes)

In [ ]:
# which words separate positive from negative? chi-square on word counts
from sklearn.feature_selection import SelectKBest, chi2

cv = CountVectorizer(stop_words="english", min_df=3)
C = cv.fit_transform(texts)
selector = SelectKBest(chi2, k=15).fit(C, labels)
words = cv.get_feature_names_out()[selector.get_support()]
print(list(words))

# check which class each word leans to
counts_neg = np.asarray(C[:100].sum(axis=0)).ravel()
counts_pos = np.asarray(C[100:].sum(axis=0)).ravel()
vocab_list = list(cv.get_feature_names_out())
for w in words:
    i = vocab_list.index(w)
    print(f"{w:12} neg {counts_neg[i]:4d}  pos {counts_pos[i]:4d}")

### Optional: 20 Newsgroups

This one downloads a dataset, so it needs internet and takes a moment.

In [ ]:
try:
    from sklearn.datasets import fetch_20newsgroups
    cats = ["rec.sport.hockey", "sci.space"]
    news = fetch_20newsgroups(subset="train", categories=cats,
                              remove=("headers", "footers", "quotes"))
    tvn = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5)
    Xn = tvn.fit_transform(news.data)
    print(Xn.shape)
except Exception as e:
    print("could not load 20 newsgroups here:", type(e).__name__)

### Dense vectors preview

Truncated SVD squeezes the sparse matrix into a few dense dimensions. Day 3 embeddings are a learned, much better version of this idea.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=0)
dense = svd.fit_transform(W)
print(np.round(dense, 2))
# the first three rows are pets, the last three are finance
# in two numbers per document the two topics already separate

## Practice

1. (Easy) Build the document-term matrix for 'dog bites man' and 'man bites dog' with CountVectorizer. What do you notice?

2. (Medium) Compute the IDF of the word 'cat' by hand for the corpus of three documents from this session, using ln((1+n)/(1+df))+1. Check against tv.idf_.

3. (Challenge) On 200 movie reviews, use chi-square to find the 10 words most associated with the negative class. Explain two of them.

In [ ]:
# your practice code here